# Caso de estudio - Supervivencia en el Titanic

# Extracción de características

Ahora trataremos parte muy importante del aprendizaje automático: la extracción de características cuantitativas a partir de los datos. Con este fin:
- Aprenderemos como las características pueden extraerse a partir de datos del mundo real.
- Veremos como extraer características numéricas a partir de datos textuales.
Además, repasaremos algunas herramientas básicas en scikit-learn que pueden utilizarse para realizar estas tareas.

## ¿Qué son características?

### Características numéricas

Recuerda que los datos en scikit-learn vienen en arrays de dos dimensiones con tamaño **n_samples** $\times$ **n_features**.

Anteriormente, vimos el dataset iris, que tienen 150 ejemplos y 4 características.

In [1]:
from sklearn.datasets import load_iris

iris = load_iris()
print(iris.data.shape)

(150, 4)


Las características son:
- Longitud de sépalo en cm
- Ancho de sépalo en cm
- Longitud de pétalo en cm
- Ancho de pétalo en cm

Las características numéricas como estas son directas: cada ejemplo contiene una lista de números con precisión real que se corresponden con las características.

### Características categóricas

¿Qué pasa si tenemos características categóricas?. Por ejemplo, imagina que disponemos del color de cada flor de iris: $color \in \{red, blue, purple\}$

Podrías estar tentado de usar algo así como i.e. *red=1, blue=2, purple=3*, pero, en general, **esto es una mala idea**. Los estimadores tienden a trabajar con la suposición de que las características numéricas se sitúan en una escala continua por lo que, en este ejemplo, 1 y 2 serían más parecidos que 1 y 3 y esto no tiene porque ser generalmente verdad.

De hecho, el ejemplo anterior es una subcategoría de las variables categóricas, en concreto, una variable nominal. Las variables nominales no tienen asociado un orden, mientras que las variables ordinales si que implican un orden. Por ejemplo, las tallas de las camisetas formarían una variable ordinal "XL > L > M > S". 

Una forma de transformar variables nominales en un formato que prevenga al estimador de asumir un orden es la llamada representación $1$-$de$-$J$ (*one-hot encoding*). Cada categoría genera su propia variable por separado.

El conjunto de características aumentado sería:
- Longitud de sépalo en cm
- Ancho de sépalo en cm
- Longitud de pétalo en cm
- Ancho de pétalo en cm
- color=purple (1.0 o 0.0)
- color=blue (1.0 o 0.0)
- color=red (1.0 o 0.0)

Observa que al usar este conjunto de características puede que los datos se representen mejor usando **matrices dispersas**, como veremos en el ejemplo de clasificación de texto que analizaremos después.

#### Utilizando DictVectorizer para codificar variables categóricas

Cuando los datos de entrada están codificados con un diccionario de tal forma que los valores son o cadenas o valores numéricos, se puede usar la clase `DictVectorizer` para obtener la expansión booleana sin tocar las características numéricas:

In [2]:
# measurements es una lista donde cada elemento es un diccionario con información sobre una ciudad y su temperatura.
measurements = [
    {'city': 'Dubai', 'temperature': 33.},
    {'city': 'London', 'temperature': 12.},
    {'city': 'San Francisco', 'temperature': 18.}
]

print(type(measurements[0]))
measurements[0]
measurements[0].keys()

<class 'dict'>


dict_keys(['city', 'temperature'])

In [3]:
from sklearn.feature_extraction import DictVectorizer

vec = DictVectorizer()
vec

,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",True
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True


In [4]:
# .fit_transform() ajusta el modelo vectorizador a los datos y transforma los diccionarios en una matriz dispersa.
# .toarray() convierte la matriz dispersa en un array denso de NumPy (ndarray). 
vec.fit_transform(measurements).toarray()

array([[ 1.,  0.,  0., 33.],
       [ 0.,  1.,  0., 12.],
       [ 0.,  0.,  1., 18.]])

In [5]:
vec.get_feature_names_out()

array(['city=Dubai', 'city=London', 'city=San Francisco', 'temperature'],
      dtype=object)

### Características derivadas

Otro tipo bastante común de características son las **características derivadas**, que son características obtenidas a partir de algún paso previo de preprocesamiento y que se supone que son más informativas que las originales. Este tipo de características pueden estar basadas en **extracción de características** y en **reducción de la dimensionalidad** (tales como PCA o aprendizaje de variedades) y pueden ser combinaciones lineales o no lineales de las características originales (como en regresión polinómica) o transformaciones más sofisticadas de las características.

### Combinando características numéricas y categóricas

Como un ejemplo de la forma en que se trabaja con datos numéricos y categóricos, vamos a realizar un ejercicio en el que predeciremos la supervivencia de los pasajeros del HMS Titanic.

Utilizaremos una versión del dataset Titatic que puede descargarse de [titanic3.xls](http://biostat.mc.vanderbilt.edu/wiki/pub/Main/DataSets/titanic3.xls). Previamente, ya hemos convertido el `.xls` a `.csv` para que sea más fácil su manipulación (como texto), de manera que los datos no fueron modificados.

Necesitamos leer todas las líneas del fichero `titanic3.csv`, ignorar la cabecera y encontrar las etiquetas (sobrevivió o murió) y los datos de entrada (características de la persona). Vamos a ver la cabecera y algunas líneas de ejemplo:

In [6]:
import os
import pandas as pd

# Obtenemos un DataFrame de pandas leyendo el archivo CSV del dataset del Titanic.
titanic = pd.read_csv(os.path.join('datasets', 'titanic3.csv'))
print(type(titanic))
print(titanic.columns)

mis_columns = titanic.columns.tolist()
print(mis_columns)


<class 'pandas.core.frame.DataFrame'>
Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket',
       'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'],
      dtype='object')
['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']


Aquí tenemos una descripción de lo que significan cada una de las variables:

```
pclass          Passenger Class
                (1 = 1st; 2 = 2nd; 3 = 3rd)
survival        Survival
                (0 = No; 1 = Yes)
name            Name
sex             Sex
age             Age
sibsp           Number of Siblings/Spouses Aboard
parch           Number of Parents/Children Aboard
ticket          Ticket Number
fare            Passenger Fare
cabin           Cabin
embarked        Port of Embarkation
                (C = Cherbourg; Q = Queenstown; S = Southampton)
boat            Lifeboat
body            Body Identification Number
home.dest       Home/Destination
```

Parece que las variables `name`, `sex`, `cabin`, `embarked`, `boat`, `body` y `homedest` son candidatas a ser variables categóricas, mientras que el resto parecen variables numéricas. Vamos a ver las primeras filas para tener un mejor conocimiento de la base de datos:

In [7]:
titanic.head() # Devuelve las 5 primeras filas del DataFrame y el tipo de dato devuelto es otro DataFrame

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


Podemos descartar directamente las columnas "boat" y "body" ya que está directamente relacionadas con que el pasajero sobreviviese. El nombre es (probablemente) único para cada persona y por tanto no es informativo. Vamos a intentar en primer lugar usar "pclass", "sibsp", "parch", "fare" y "embarked" como características:

In [8]:
labels = titanic.survived.values # Convertimos la columna 'survived' en un array de NumPy
print(labels) # Imprime las etiquetas (valores) de supervivencia, es decir, imprime la clase para cada patron.
features = titanic[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']] # Seleccionamos las características relevantes


[1 1 0 ... 0 0 0]


In [9]:
features.head() # Imprime las primeras filas de las características seleccionadas

,pclass,sex,age,sibsp,parch,fare,embarked
0,1,female,29.0000,0,0,211.3375,S
1,1,male,0.9167,1,2,151.5500,S
2,1,female,2.0000,1,2,151.5500,S
3,1,male,30.0000,1,2,151.5500,S
4,1,female,25.0000,1,2,151.5500,S


En principio, los datos ahora solo contienen características útiles, pero no están en un formato que los algoritmos de aprendizaje automático puedan entender. Necesitamos transformar las cadenas "male" y "female" en variables binarias que indiquen el género y lo mismo para `embarked`.Podemos hacer esto usando la función ``get_dummies`` de pandas:

In [10]:
pd.get_dummies(features).head() # Convierte las variables categóricas en variables esquema 1 de J 

,pclass,age,sibsp,parch,fare,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,1,29.0000,0,0,211.3375,True,False,False,False,True
1,1,0.9167,1,2,151.5500,False,True,False,False,True
2,1,2.0000,1,2,151.5500,True,False,False,False,True
3,1,30.0000,1,2,151.5500,False,True,False,False,True
4,1,25.0000,1,2,151.5500,True,False,False,False,True


Esta transformación ha codificado bien las columnas de cadenas. Sin embargo, parece que la variable ``pclass`` también es una variable categórica. Podemos listar de forma explícita las variables que queremos codificar utilizando el parámetro ``columns`` para incluir ``pclass``:

In [11]:
features_dummies = pd.get_dummies(features, columns=['pclass', 'sex', 'embarked']) # Convierte concretamente las variables categóricas pclass, sex y embarked en variables esquema 1 de J
features_dummies.head(n=16) # Muestra las primeras 16 filas del DataFrame resultante

,age,sibsp,parch,fare,pclass_1,pclass_2,pclass_3,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,29.0000,0,0,211.3375,True,False,False,True,False,False,False,True
1,0.9167,1,2,151.5500,True,False,False,False,True,False,False,True
2,2.0000,1,2,151.5500,True,False,False,True,False,False,False,True
3,30.0000,1,2,151.5500,True,False,False,False,True,False,False,True
4,25.0000,1,2,151.5500,True,False,False,True,False,False,False,True
5,48.0000,0,0,26.5500,True,False,False,False,True,False,False,True
6,63.0000,1,0,77.9583,True,False,False,True,False,False,False,True
7,39.0000,0,0,0.0000,True,False,False,False,True,False,False,True
8,53.0000,2,0,51.4792,True,False,False,True,False,False,False,True
9,71.0000,0,0,49.5042,True,False,False,False,True,True,False,False


In [12]:
#También podríamos hacerlo con DictVectorizer
from sklearn.feature_extraction import DictVectorizer

diccionario = features.to_dict('records')
vec = DictVectorizer()
dataset = vec.fit_transform(diccionario)
print(dataset.todense())

[[29.      0.      0.     ...  1.      0.      0.    ]
 [ 0.9167  0.      0.     ...  0.      1.      1.    ]
 [ 2.      0.      0.     ...  1.      0.      1.    ]
 ...
 [26.5     0.      1.     ...  0.      1.      0.    ]
 [27.      0.      1.     ...  0.      1.      0.    ]
 [29.      0.      0.     ...  0.      1.      0.    ]]


In [13]:
data = features_dummies.values
print(data.shape)
print(data.dtype)
data

(1309, 12)
object


array([[29.0, 0, 0, ..., False, False, True],
       [0.9167, 1, 2, ..., False, False, True],
       [2.0, 1, 2, ..., False, False, True],
       ...,
       [26.5, 0, 0, ..., True, False, False],
       [27.0, 0, 0, ..., True, False, False],
       [29.0, 0, 0, ..., False, False, True]],
      shape=(1309, 12), dtype=object)

In [14]:
# Comprobamos que hay valores perdidos, tendremos que aplicar un Imputer
import numpy as np
# np.isnan(data).any()
np.isnan(data.astype(float)).any()

np.True_

Una vez hemos hecho el trabajo de duro de cargar los datos, evaluar un clasificador con estos datos es directo. Vamos a ver que rendimiento obtenemos con el clasificador más simple, `DummyClassifier('most_frequent')`, que es equivalente al `ZeroR`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Dividimos los datos en conjuntos de entrenamiento y prueba
train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, random_state=0)

# Se utiliza SimpleImputer para rellenar los valores perdidos con la media de cada columna
imp = SimpleImputer()
# IMPORTANTE: Para manejar valores faltantes: ajusta el imputador con los datos de entrenamiento y transforma tanto entrenamiento como prueba. Esto evita “filtrar” información del conjunto de prueba (gotcha común en ML), ya que el imputador aprende solo del entrenamiento.
imp.fit(train_data)
train_data_finite = imp.transform(train_data)
test_data_finite = imp.transform(test_data)

In [16]:
np.isnan(train_data_finite).any()

np.False_

In [17]:
from sklearn.dummy import DummyClassifier

clf = DummyClassifier(strategy='most_frequent')
clf.fit(train_data_finite, train_labels)
print("Accuracy: %f"
      % clf.score(test_data_finite, test_labels))

Accuracy: 0.634146


<div class="alert alert-success">
    <b>EJERCICIO</b>:
     <ul>
      <li>
      Intenta ejecutar el problema de clasificación anterior pero usando ``LogisticRegression`` y ``RandomForestClassifier`` en lugar de ``DummyClassifier``
      </li>
      <li>
      Prueba a cambiar el conjunto de características considerado. ¿Consigues mejorar los resultados?
      </li>
    </ul>
</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Dividimos los datos en conjuntos de entrenamiento y prueba
train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, test_size=0.2, stratify=labels, random_state=0)

# Se utiliza SimpleImputer para rellenar los valores perdidos con la media de cada columna
imp = SimpleImputer()
imp.fit(train_data)
# IMPORTANTE: Para manejar valores faltantes: ajusta el imputador con los datos de entrenamiento y transforma tanto entrenamiento como prueba. Esto evita “filtrar” información del conjunto de prueba (gotcha común en ML), ya que el imputador aprende solo del entrenamiento.
train_data_finite = imp.transform(train_data)
test_data_finite = imp.transform(test_data)

np.isnan(train_data_finite).any()

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0, max_iter=1000)
clf.fit(train_data_finite, train_labels)
test_labels_pred = clf.predict(test_data_finite)

# Score espera de segundo parámetro las etiquetas correctas, predice y hace la comparación de esa predicción con dichas etiquetas correctas
print("Accuracy using train_x y train_y: %f" % clf.score(train_data_finite, train_labels))
print("Accuracy using test_x y test_y: %f" % clf.score(test_data_finite, test_labels))

# Esto estaría mal, porque le estamos diciendo que prediga y compare con las etiqeutas predichas, por eso es 1
print("Accuracy using test_x y test_predicho (MAL_HECHO): %f" % clf.score(test_data_finite, test_labels_pred))

Accuracy using train_x y train_y: 0.787966
Accuracy using test_x y test_y: 0.805344
Accuracy using test_x y test_predicho (MAL): 1.000000


In [ ]:
# Para coger los datos sobre la clase a predecir, que es superviviencia
labels = titanic.survived.values # Convertimos la columna 'survived' en un array de NumPy
# Imprime las etiquetas (valores) de supervivencia, es decir, imprime la clase para cada patron.
print(labels)
# Vamos a cambiar las características seleccionadas para ajustar el modelo y hacer la predicción.
features = titanic[['pclass', 'sex', 'age', 'fare', 'embarked']] 
# Imprime las primeras filas de las características seleccionadas
features.head() 


[1 1 0 ... 0 0 0]


,pclass,sex,age,fare,embarked
0,1,female,29.0000,211.3375,S
1,1,male,0.9167,151.5500,S
2,1,female,2.0000,151.5500,S
3,1,male,30.0000,151.5500,S
4,1,female,25.0000,151.5500,S


In [34]:
# Convierte las variables categóricas en variables esquema 1 de J 
#pd.get_dummies(features).head()
features_dummies = pd.get_dummies(features, columns=['pclass', 'sex',  'embarked']) 
# Muestra las primeras 16 filas del DataFrame resultante
features_dummies.head(n=16) 
# Muestra los datos pero ya en forma de ndarray
data = features_dummies.values
data

array([[29.0, 211.3375, True, ..., False, False, True],
       [0.9167, 151.55, True, ..., False, False, True],
       [2.0, 151.55, True, ..., False, False, True],
       ...,
       [26.5, 7.225, False, ..., True, False, False],
       [27.0, 7.225, False, ..., True, False, False],
       [29.0, 7.875, False, ..., False, False, True]],
      shape=(1309, 10), dtype=object)

In [35]:
# Comprobamos si hay valores perdidos
np.isnan(data.astype(float)).any()

# Sustituimos los valores perdidos con un Imputer, pero antes partimos el conjunto de datos

# Dividimos los datos en conjuntos de entrenamiento y prueba
train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, random_state=0)

# Se utiliza SimpleImputer para rellenar los valores perdidos con la media de cada columna
imp = SimpleImputer()
# IMPORTANTE: Para manejar valores faltantes: ajusta el imputador con los datos de entrenamiento y transforma tanto entrenamiento como prueba. Esto evita “filtrar” información del conjunto de prueba (gotcha común en ML), ya que el imputador aprende solo del entrenamiento.
imp.fit(train_data)
train_data_finite = imp.transform(train_data)
test_data_finite = imp.transform(test_data)

# Ya no hay datos perdidos
np.isnan(train_data_finite).any()



np.False_

In [36]:
clf = LogisticRegression(random_state=0, max_iter=1000)
clf.fit(train_data_finite, train_labels)
test_labels_pred = clf.predict(test_data_finite)

# Score espera de segundo parámetro las etiquetas correctas, predice y hace la comparación de esa predicción con dichas etiquetas correctas
print("Accuracy using train_x y train_y: %f" % clf.score(train_data_finite, train_labels))
print("Accuracy using test_x y test_y: %f" % clf.score(test_data_finite, test_labels))

Accuracy using train_x y train_y: 0.782875
Accuracy using test_x y test_y: 0.786585
